In [1]:
import pandas as pd
import numpy as np
import fastf1
import requests

In [2]:
def get_fp1_best(FP1):

    laps = FP1.laps.pick_quicklaps()

    best = laps.groupby("Driver")["LapTime"].min().dt.total_seconds().reset_index()

    return best

In [3]:
def get_fp2_best(FP2):

    laps = FP2.laps.pick_quicklaps()

    best = laps.groupby("Driver")["LapTime"].min().dt.total_seconds().reset_index()

    return best

In [4]:
def get_fp3_best(FP3):

    laps = FP3.laps.pick_quicklaps()

    best = laps.groupby("Driver")["LapTime"].min().dt.total_seconds().reset_index()

    return best

In [5]:
def get_sector_times (FP2):

    laps = FP2.laps

    sector_times = laps.groupby("Driver")[[
    "Sector1Time",
    "Sector2Time",
    "Sector3Time"
    ]].mean().apply(lambda x: x.dt.total_seconds()).reset_index()

    return sector_times

In [6]:
def get_average_laptime (FP2):

    laps = FP2.laps

    avg_lap = laps.groupby("Driver")["LapTime"].mean().dt.total_seconds().reset_index()

    return avg_lap

In [7]:
def get_stint(FP2):

    laps = FP2.laps

    stints = (
        laps.groupby(["Driver","Stint"])
        .size()
        .reset_index(name="LapCount")
    )

    longest_stint = (
        stints.groupby("Driver")["LapCount"]
        .max()
        .reset_index(name="LongestStint")
    )

    return longest_stint

In [8]:
def get_tyre_compound (FP2):

    laps = FP2.laps

    tyres = laps.groupby("Driver")["Compound"].agg(lambda x: x.mode()[0]).reset_index()

    return tyres

In [9]:
def get_qualifying_data(Qualify):

    results = Qualify.results.copy()

    qual_time = results[["Q3", "Q2", "Q1"]].bfill(axis=1).iloc[:, 0]

    qual = results[["Abbreviation"]].copy()
    qual["Qualifying_Time(s)"] = qual_time.dt.total_seconds()

    qual.rename(columns={"Abbreviation": "Driver"}, inplace=True)

    return qual

In [10]:
def get_race_results(Result):

    race = Result.results[["Abbreviation","Position"]]
    race.rename(columns={"Abbreviation":"Driver"},inplace=True)

    return race

In [25]:
def Rain(API_KEY,lat,lon,forecast_time):
    url = f"http://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_KEY}&units=metric"

    response = requests.get(url)
    data = response.json()

    if "list" not in data:
        print("API ERROR:", data)
    else:

        forecast_data = next(
            (f for f in data["list"] if f["dt_txt"] == forecast_time),
            None
        )
        if forecast_data:
            rain_probability = forecast_data.get("pop", 0)
            temperature = forecast_data["main"]["temp"]
        else:
            forecast_data = data["list"][0]
            rain_probability = forecast_data.get("pop", 0)
            temperature = forecast_data["main"]["temp"]

        return rain_probability,temperature